# DataLoader multimodal de galaxias

Esta libreta explica cómo usar la infraestructura de datos para los experimentos de clasificación morfológica. No entrena modelos ni modifica el manifest, los splits o las cachés.

El contrato permite activar cualquier combinación de tres modalidades:

- `image`: cutout RGB elegido por FOV.
- `spectrum`: tensor espectral preprocesado desde la caché.
- `metadata`: variables numéricas permitidas del manifest.

El objetivo binario permanece fijo: `temprana = 0.0` y `tardia = 1.0`, para usarlo con `BCEWithLogitsLoss`.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import torch


def project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto.")


ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataloader.multimodal import build_dataloaders

MANIFEST_PATH = ROOT / "data/galaxy_manifest.csv"
# Este dataset esta alineado con la disponibilidad de cache de espectros para FoV
SPLITS_PATH = ROOT / "data/processed/splits_seed42_obj-label_FoV.csv"
SPECTRA_ROOT = ROOT / "data/processed/spectra"

print(f"Proyecto: {ROOT}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Splits: {SPLITS_PATH}")

Proyecto: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias
Manifest: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias/data/galaxy_manifest.csv
Splits: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias/data/processed/splits_seed42_obj-label_FoV.csv


## 1. Qué datos ve el loader

El manifest es el índice central: contiene la etiqueta, rutas y disponibilidad de cada FOV, además de variables físicas y observacionales. El archivo de splits solo asigna cada `name` a `train`, `validation` o `test`.

`build_dataloaders` une ambos por `name`, verifica que sus etiquetas coincidan y conserva una sola partición por galaxia. Las variantes FOV de un mismo objeto nunca se separan entre conjuntos.

In [3]:
manifest = pd.read_csv(MANIFEST_PATH)
splits = pd.read_csv(SPLITS_PATH)

fov_columns = [column for column in manifest.columns if column.startswith("fov_") and column.count("_") == 1]
print(f"Filas del manifest: {len(manifest):,}")
print(f"Filas del split: {len(splits):,}")
print("\nDistribución del split:")
display(splits.groupby(["split", "label"], observed=True).size().unstack(fill_value=0))
print("\nDisponibilidad por FOV dentro de la cohorte del split:")
display(
    splits[["name"]]
    .merge(manifest[["name", *fov_columns]], on="name", validate="one_to_one")[fov_columns]
    .sum()
    .rename("imágenes disponibles")
    .to_frame()
)

Filas del manifest: 10,014
Filas del split: 9,351

Distribución del split:


label,tardia,temprana
split,,
test,875,528
train,4078,2467
validation,874,529



Disponibilidad por FOV dentro de la cohorte del split:


,imágenes disponibles
fov_+3,9351
fov_+2,9351
fov_+1,9351
fov_0,9351
fov_-1,9351
fov_-2,9351
fov_-3,9351


## 2. Imágenes: FOV, augmentations y normalización

La modalidad `image` usa una columna FOV configurable, por ejemplo `fov_0_path`. Durante entrenamiento se aplican solamente transformaciones geométricas que preservan la clase morfológica: rotación aleatoria de $0^\circ$, $90^\circ$, $180^\circ$ o $270^\circ$, flip horizontal y flip vertical.

Validation y test no llevan augmentación: convierten explícitamente a RGB, hacen resize a `256 × 256` y convierten a tensor. `normalization="none"` mantiene los JPEG en `[0, 1]`; `normalization="imagenet"` se usa si el encoder es una ResNet con pesos ImageNet.

In [4]:
image_only_config = {
    "data": {
        "manifest": MANIFEST_PATH,
        "splits": SPLITS_PATH,
        "modalities": ["image"],
        "image": {
            "fov_column": "fov_0",
            "size": 256,
            "normalization": "none",  # Cambiar a "imagenet" para un encoder preentrenado.
        },
    },
    "training": {
        "batch_size": 32,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    },
}

# train_loader, validation_loader, test_loader = build_dataloaders(image_only_config)

## 3. Metadatos: selección y normalización sin leakage

Los metadatos se vuelven un tensor `metadata` y una máscara booleana `metadata_mask`. La máscara marca los valores originalmente disponibles; la imputación permite que el tensor permanezca numérico.

La mediana para imputar, la media y la desviación estándar se ajustan solo con `train`. Validation y test reutilizan exactamente esos parámetros. Nunca se permiten `label`, `Ttype`, `Type`, `name`, `objra` u `objdec` como predictors: introducirlos filtraría la respuesta o un identificador al modelo.

In [5]:
image_metadata_config = {
    **image_only_config,
    "data": {
        **image_only_config["data"],
        "modalities": ["image", "metadata"],
        "metadata": {
            "features": ["nsa_z", "LogMass", "PETRO_TH90", "incl"],
        },
    },
}

# train_loader, validation_loader, test_loader = build_dataloaders(image_metadata_config)
# Los batches incluirán: id, label, image, metadata y metadata_mask.

## 4. Espectros: caché, malla, máscara y normalización

El loader no abre FITS por época. Primero se crea una caché reproducible que realiza una vez por galaxia:

1. Lee `WAVE`, `FLUX_1RE` y/o `FLUX_FoV`.
2. Convierte el eje observado a reposo con $\lambda_\mathrm{rest}=\lambda_\mathrm{obs}/(1+z)$.
3. Normaliza con la mediana de continuo en $5400$–$5500$ Å de reposo.
4. Interpola a una malla común de $3600$–$9000$ Å con 4,096 muestras, sin extrapolar.
5. Rellena fuera de cobertura con cero y entrega `spectrum_mask=False` en esos puntos.

La máscara evita que ceros de relleno se interpreten como flujo físico. `per_channel` normaliza cada apertura de forma independiente; `shared` usa una sola escala por galaxia y conserva la relación de amplitud entre aperturas.

In [6]:
cache_candidates = sorted(SPECTRA_ROOT.glob("spectra_*")) if SPECTRA_ROOT.exists() else []
print("Cachés espectrales encontradas:")
for candidate in cache_candidates:
    print("-", candidate.relative_to(ROOT))

# Después de crear una caché, seleccionar exactamente una ruta impresa por:
# .venv/bin/python -m src.dataloader.cache_spectra
SPECTRUM_CACHE_DIR = cache_candidates[0] if len(cache_candidates) == 1 else None

if SPECTRUM_CACHE_DIR is None:
    print("Configure SPECTRUM_CACHE_DIR después de generar o elegir una caché.")
else:
    print(f"Caché seleccionada: {SPECTRUM_CACHE_DIR.relative_to(ROOT)}")

Cachés espectrales encontradas:
- data/processed/spectra/spectra_cache_1RE
- data/processed/spectra/spectra_cache_1RE_FoV
- data/processed/spectra/spectra_cache_FoV
Configure SPECTRUM_CACHE_DIR después de generar o elegir una caché.


In [7]:
spectrum_only_config = {
    "data": {
        "manifest": MANIFEST_PATH,
        "splits": SPLITS_PATH,
        "modalities": ["spectrum"],
        "spectrum": {
            "cache_dir": SPECTRUM_CACHE_DIR,
            # La selección de canales y la normalización quedan fijadas por la caché.
        },
    },
    "training": {"batch_size": 32, "num_workers": 0, "pin_memory": torch.cuda.is_available()},
}

multimodal_config = {
    **image_metadata_config,
    "data": {
        **image_metadata_config["data"],
        "modalities": ["image", "spectrum", "metadata"],
        "spectrum": spectrum_only_config["data"]["spectrum"],
    },
}

# Para usar solo una apertura, crear la caché con:
# .venv/bin/python -m src.dataloader.cache_spectra --channels FLUX_1RE
# o bien:
# .venv/bin/python -m src.dataloader.cache_spectra --channels FLUX_FoV
# Para ambas aperturas y escala compartida:
# .venv/bin/python -m src.dataloader.cache_spectra --channels FLUX_1RE FLUX_FoV --normalization shared

## 5. Smoke test de un DataLoader

El smoke test no entrena nada. Construye los tres loaders, toma un batch de entrenamiento y verifica el contrato antes de implementar un modelo:

- train/validation/test existen y conservan sus tamaños.
- `label` es `float32`, con valores binarios.
- Las modalidades activas tienen las formas esperadas.
- Los tensores son finitos.
- La máscara espectral tiene forma `[B, L]` y acompaña al espectro `[B, C, L]`.

Ejecutar primero con `image_only_config` o `image_metadata_config`. Las configuraciones con espectro requieren que `SPECTRUM_CACHE_DIR` apunte a una caché creada.

In [ ]:
def summarize_batch(batch: dict[str, object]) -> None:
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, finite={bool(torch.isfinite(value).all()) if value.is_floating_point() else 'n/a'}")
        else:
            print(f"{key}: {type(value).__name__}, length={len(value)}")


def run_smoke_test(config: dict[str, object]) -> None:
    train_loader, validation_loader, test_loader = build_dataloaders(config)
    batch = next(iter(train_loader))
    print("Tamaños de dataset:", {
        "train": len(train_loader.dataset),
        "validation": len(validation_loader.dataset),
        "test": len(test_loader.dataset),
    })
    summarize_batch(batch)

    assert batch["label"].dtype == torch.float32
    assert set(batch["label"].unique().tolist()).issubset({0.0, 1.0})
    if "image" in batch:
        assert batch["image"].ndim == 4 and batch["image"].shape[1] == 3
    if "metadata" in batch:
        assert batch["metadata"].shape == batch["metadata_mask"].shape
    if "spectrum" in batch:
        assert batch["spectrum"].ndim == 3
        assert batch["spectrum_mask"].shape == batch["spectrum"].shape[:1] + batch["spectrum"].shape[2:]
    print("Smoke test superado.")


# Ejecutar una sola línea según la modalidad a probar:
# run_smoke_test(image_only_config)
# run_smoke_test(image_metadata_config)

# Solamente espectros
# run_smoke_test(multimodal_config)

Tamaños de dataset: {'train': 6545, 'validation': 1403, 'test': 1403}
id: list, length=32
label: shape=(32,), dtype=torch.float32, finite=True
spectrum: shape=(32, 1, 4096), dtype=torch.float32, finite=True
spectrum_mask: shape=(32, 4096), dtype=torch.bool, finite=n/a
Smoke test superado.


In [10]:
run_smoke_test(image_only_config)

Tamaños de dataset: {'train': 6545, 'validation': 1403, 'test': 1403}
id: list, length=32
label: shape=(32,), dtype=torch.float32, finite=True
image: shape=(32, 3, 256, 256), dtype=torch.float32, finite=True
Smoke test superado.


In [ ]:
# Ejecutar prueba rápida con extracción de metadatos de imagen
run_smoke_test(image_metadata_config)

Tamaños de dataset: {'train': 6545, 'validation': 1403, 'test': 1403}
id: list, length=32
label: shape=(32,), dtype=torch.float32, finite=True
image: shape=(32, 3, 256, 256), dtype=torch.float32, finite=True
metadata: shape=(32, 4), dtype=torch.float32, finite=True
metadata_mask: shape=(32, 4), dtype=torch.bool, finite=n/a
Smoke test superado.


In [12]:
# Ejecutar prueba rápida con solamente espectros
spectrum_only_config["data"]["spectrum"]["cache_dir"] = SPECTRA_ROOT / "spectra_cache_FoV"
run_smoke_test(spectrum_only_config)

Tamaños de dataset: {'train': 6545, 'validation': 1403, 'test': 1403}
id: list, length=32
label: shape=(32,), dtype=torch.float32, finite=True
spectrum: shape=(32, 1, 4096), dtype=torch.float32, finite=True
spectrum_mask: shape=(32, 4096), dtype=torch.bool, finite=n/a
Smoke test superado.


In [13]:
# Ejecutar prueba rápida con multimodalidad (imágenes + metadatos + espectros)
multimodal_config["data"]["spectrum"]["cache_dir"] = SPECTRA_ROOT / "spectra_cache_FoV"
run_smoke_test(multimodal_config)

Tamaños de dataset: {'train': 6545, 'validation': 1403, 'test': 1403}
id: list, length=32
label: shape=(32,), dtype=torch.float32, finite=True
image: shape=(32, 3, 256, 256), dtype=torch.float32, finite=True
metadata: shape=(32, 4), dtype=torch.float32, finite=True
metadata_mask: shape=(32, 4), dtype=torch.bool, finite=n/a
spectrum: shape=(32, 1, 4096), dtype=torch.float32, finite=True
spectrum_mask: shape=(32, 4096), dtype=torch.bool, finite=n/a
Smoke test superado.
